In [44]:
import numpy as np
import pandas as pd


In [45]:
dtype_map = {
    "Task Id": "string",
    "FPOGV": "Int64", "FPOGID": "Int64",
    "LPMMV": "Int64", "RPMMV": "Int64",
    "BKID": "Int64",  "BKDUR": "float64", 
    "AOI": "string"
}
fixations_onoff = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fix_onoff_task2.csv", dtype=dtype_map)
gaze_onoff = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/gaze_onoff_task2.csv", dtype=dtype_map)

In [48]:
import numpy as np
import pandas as pd

TICK_HZ = 10_000_000  # TIMETICK(f=10000000)
EPS = 1e-9

# ----------------------
# Helper functions
# ----------------------
def _mad(x):
    x = pd.Series(x, dtype="float64").dropna()
    if x.empty: return np.nan
    med = x.median()
    return (x - med).abs().median()

def _count_sign_changes(arr: np.ndarray) -> int:
    """Count sign flips in a 1D series (ignoring zeros)."""
    if arr.size < 3:
        return 0
    s = np.sign(arr.astype(float))
    for i in range(1, s.size):
        if s[i] == 0:
            s[i] = s[i-1]
    s = s[s != 0]
    if s.size < 2:
        return 0
    return int(np.sum(s[1:] * s[:-1] < 0))

def _shannon_entropy(p_vec: np.ndarray) -> float:
    p = np.asarray(p_vec, float)
    p = p[np.isfinite(p)]
    p = p[p > 0]
    if p.size == 0:
        return np.nan
    p = p / p.sum()
    return float(-(p * np.log2(p)).sum())

def _aoi_bucket(x):
    """Map raw AOI strings to coarse buckets TEXT/MAP/OTHER."""
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    if "text" in s or "paragraph" in s or "title" in s:
        return "TEXT"
    if "map" in s or "legend" in s or "marker" in s:
        return "MAP"
    return "OTHER"

# ----------------------
# Core feature extractor
# ----------------------
def window_features_task2(
    df: pd.DataFrame,
    *,
    win_sec: float = 3.0,
    min_valid_frac: float = 0.60,
    left_x_thresh: float = 0.45,
    right_x_thresh: float = 0.55,
) -> pd.DataFrame:
    """
    Compute only the requested Task-2 features:
      fix_rate_per_s, fix_median_dur, gaze_dispersion_total,
      saccade_mean_mag, saccade_rate_per_s,
      dir_change_rate, axis_ratio,
      dwell_text_ratio, dwell_map_ratio, aoi_entropy,
      trans_text_to_map, trans_map_to_text,
      x_mean, x_dir_change_rate, left_ratio, right_ratio, x_entropy
    """
    keys = ["Video Name (folder)", "User ID (file)", "win_id"]

    need_cols = {"FPOGX","FPOGY","FPOGV"}
    has_coords = need_cols.issubset(df.columns)
    has_aoi    = "AOI" in df.columns

    tick_col = "TIMETICK(f=10000000)"

    def _agg(w: pd.DataFrame) -> pd.Series:
        out = {}
        gv = (w["FPOGV"] == 1) if "FPOGV" in w else pd.Series(True, index=w.index)
        valid_frac = float(gv.mean()) if len(w) else 0.0

        # -----------------
        # Fixation features
        # -----------------
        if "FPOGID" in w:
            fix_count = w["FPOGID"].dropna().nunique()
        else:
            fix_count = np.nan
        out["fix_rate_per_s"]  = (fix_count / win_sec) if pd.notna(fix_count) else np.nan
        out["fix_median_dur"]  = (w["FPOGD"].median() if "FPOGD" in w else np.nan)

        # -----------------
        # Gaze dispersion
        # -----------------
        if has_coords:
            gx = w.loc[gv, "FPOGX"].std()
            gy = w.loc[gv, "FPOGY"].std()
            out["gaze_dispersion_total"] = (
                np.sqrt((0 if pd.isna(gx) else gx**2) + (0 if pd.isna(gy) else gy**2))
                if (not pd.isna(gx) or not pd.isna(gy)) else np.nan
            )
        else:
            out["gaze_dispersion_total"] = np.nan

        # -----------------
        # Saccade features
        # -----------------
        if "SACCADE_MAG" in w and gv.any():
            mags = w.loc[gv, "SACCADE_MAG"].dropna().to_numpy()
            out["saccade_mean_mag"] = float(np.mean(mags)) if mags.size else np.nan
            # heuristic saccade count
            mag = w.loc[gv, "SACCADE_MAG"].fillna(0.0).to_numpy()
            rising = (mag[1:] > 0) & (mag[:-1] <= 0) if mag.size >= 2 else np.array([], dtype=bool)
            sacc_count = int(rising.sum())
            out["saccade_rate_per_s"] = sacc_count / max(win_sec, EPS)
        else:
            out["saccade_mean_mag"] = np.nan
            out["saccade_rate_per_s"] = np.nan

        # -----------------
        # Axis ratio + dir change
        # -----------------
        if has_coords and valid_frac >= min_valid_frac:
            x = w.loc[gv, "FPOGX"].to_numpy(float)
            y = w.loc[gv, "FPOGY"].to_numpy(float)
            sx, sy = float(np.nanstd(x)), float(np.nanstd(y))
            out["axis_ratio"] = (sy / (sx + EPS)) if np.isfinite(sx) and np.isfinite(sy) else np.nan
            v = np.diff(y) if np.nan_to_num(sy) >= np.nan_to_num(sx) else np.diff(x)
            out["dir_change_rate"] = _count_sign_changes(v) / max(win_sec, EPS) if v.size else np.nan
        else:
            out["axis_ratio"] = np.nan
            out["dir_change_rate"] = np.nan

        # -----------------
        # AOI-based dwell & transitions
        # -----------------
        out["dwell_text_ratio"]  = np.nan
        out["dwell_map_ratio"]   = np.nan
        out["aoi_entropy"]       = np.nan
        out["trans_text_to_map"] = np.nan
        out["trans_map_to_text"] = np.nan

        if has_aoi and valid_frac >= min_valid_frac:
            aoi_bucket = w["AOI"].map(_aoi_bucket)
            aoi_bucket = aoi_bucket.where(gv, other=np.nan)
            n_valid = int((~aoi_bucket.isna()).sum())
            if n_valid > 0:
                text_mask  = aoi_bucket.eq("TEXT")
                map_mask   = aoi_bucket.eq("MAP")
                other_mask = aoi_bucket.eq("OTHER")
                dt = float(text_mask.sum())  / n_valid
                dm = float(map_mask.sum())   / n_valid
                do = float(other_mask.sum()) / n_valid
                out["dwell_text_ratio"] = dt
                out["dwell_map_ratio"]  = dm
                out["aoi_entropy"]      = _shannon_entropy(np.array([dt, dm, do]))

                seq = aoi_bucket.dropna().to_numpy(str)
                if seq.size >= 2:
                    t2m = m2t = 0
                    for a, b in zip(seq[:-1], seq[1:]):
                        if a != b:
                            if   a == "TEXT" and b == "MAP":  t2m += 1
                            elif a == "MAP"  and b == "TEXT": m2t += 1
                    out["trans_text_to_map"] = t2m / max(win_sec, EPS)
                    out["trans_map_to_text"] = m2t / max(win_sec, EPS)

        # -----------------
        # X-split features
        # -----------------
        out["x_mean"]             = np.nan
        out["x_dir_change_rate"]  = np.nan
        out["left_ratio"]         = np.nan
        out["right_ratio"]        = np.nan
        out["x_entropy"]          = np.nan

        if has_coords and valid_frac >= min_valid_frac:
            x = w.loc[gv, "FPOGX"].to_numpy(float)
            out["x_mean"] = float(np.nanmean(x)) if x.size else np.nan
            left_mask  = x < left_x_thresh
            right_mask = x > right_x_thresh
            left_ratio  = float(np.mean(left_mask))  if x.size else np.nan
            right_ratio = float(np.mean(right_mask)) if x.size else np.nan
            mid_ratio   = (1.0 - (left_ratio + right_ratio)) if np.all(np.isfinite([left_ratio,right_ratio])) else np.nan
            out["left_ratio"]  = left_ratio
            out["right_ratio"] = right_ratio
            out["x_entropy"]   = _shannon_entropy(np.array([left_ratio, mid_ratio, right_ratio])) if np.isfinite(mid_ratio) else np.nan
            dx = np.diff(x)
            out["x_dir_change_rate"] = _count_sign_changes(dx) / max(win_sec, EPS) if dx.size else np.nan

        return pd.Series(out)

    d = (
        df.dropna(subset=["win_id"])
          .groupby(keys, dropna=False)
          .apply(_agg)
          .reset_index()
    )

    return d

In [52]:
raw = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fix_onoff_task2.csv")
win_len = 3.0  # seconds
ref = build_shared_anchor(raw, tick_col="TIMETICK(f=10000000)")
raw = add_window_id_by_timetick(raw, win_sec=win_len, shared_ref=ref)
raw_refined = refine_window_labels_by_fraction(raw, win_col="win_id", label_col="label", on_hi=0.80, off_lo=0.20)
#
# # If you keep fixations and gaze in the same `raw_refined`, you can pass the same df twice:
features_task2 = combine_fix_gaze_scroll(raw_refined, raw_refined, win_sec=win_len, aoi_role_map=None)
display(features_task2.head())

/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_13345/338191820.py:202: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_13345/338191820.py:270: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_13345/338191820.py:340: DeprecationWarning: DataFrameGroupBy.apply 

,Video Name (folder),User ID (file),Task Id,type,label,phase,block_id,win_id,fix_efficiency,fix_aoi_ratio,...,saccade_mean_mag,saccade_dir_var,saccade_circ_var,blink_rate,valid_gaze_ratio,vert_frac,net_vert_disp,prop_down_saccades,fix_y_over_x,trans_content_to_footer
0,P10_S1_ITA,User 0,2.1,BOUNCE,ON,ON,P10_S1_ITA|User 0|2.1,1,32.805170,0.0,...,1042.440127,8314.539306,0.601162,0.333333,1.0,0.854620,-1.22625,0.285714,8.064481,0.0
1,P10_S1_ITA,User 0,2.1,BOUNCE,ON,ON,P10_S1_ITA|User 0|2.1,2,37.925805,0.0,...,576.496207,11128.843537,0.843592,0.333333,1.0,0.834794,1.49825,0.555556,6.409059,0.0
2,P10_S1_ITA,User 0,2.1,BOUNCE,ON,ON,P10_S1_ITA|User 0|2.1,3,32.444809,0.0,...,1291.728777,7117.669089,0.700828,0.333333,1.0,0.818267,-0.08619,0.285714,5.614732,0.0
3,P10_S1_ITA,User 0,2.1,Unknown,OFF,OFF_POST,P10_S1_ITA|User 0|2.1,4,32.172638,0.0,...,374.990368,10208.664145,0.774886,1.000000,1.0,0.740624,0.84776,0.625000,3.206816,0.0
4,P10_S1_ITA,User 0,2.1,Unknown,OFF,OFF_POST,P10_S1_ITA|User 0|2.1,5,16.111910,0.0,...,322.062503,1109.866950,0.109541,0.333333,1.0,0.670464,-0.22012,0.000000,1.979038,0.0


In [55]:
features_task2 = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/improving models/outputs2/fix_wid_t2.csv")

In [56]:
# --- your mappings ---
task_mapping = {
    '1.1': 'C011', '1.2': 'C100', '1.3': 'C000', '1.4': 'Cnull', '1.5': 'C100',
    '1.6': 'C110', '1.7': 'C011', '1.8': 'C001', '1.9': 'C111', '2.1': 'B000',
    '2.2': 'Bnull', '2.3': 'B112', '2.4': 'B113', '2.5': 'B012', '2.6': 'B013',
    '2.7': 'B002', '2.8': 'B102', '2.9': 'B103', '3.1': 'A000', '3.2': 'A100',
    '3.3': 'A010', '3.4': 'A101', '3.5': 'A001', '3.6': 'A111', '3.7': 'A110',
    '3.8': 'A011', '3.9': 'Anull'
}

animation_mapping = {
    'A': {
        'type': 'FADE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
    'B': {
        'type': 'BOUNCE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': {0: 'Left to Right', 1: 'Top to Bottom', 2: 'Bouncing', 3: 'Pulsing'},
    },
    'C': {
        'type': 'SCROLL',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
}

def decode_from_code(code: str):
    """
    Return dict with decoded fields from a code like 'C011', 'B112', or '*null'.
    """
    if not isinstance(code, str) or "null" in code.lower():
        return {
            "anim_family": np.nan,
            "anim_code": code if isinstance(code, str) else np.nan,
            "anim_type": "NO-ANIM",
            "anim_delay": np.nan,
            "anim_ease": np.nan,
            "anim_direction": np.nan,
            "block_anim": False,
        }

    fam = code[0]                 # 'A' / 'B' / 'C'
    spec = code[1:]               # e.g. '011' or '112'
    meta = animation_mapping.get(fam, {})

    # defaults
    delay = ease = direction = np.nan

    # parse indices safely
    try:
        i_delay = int(spec[0]) if len(spec) >= 1 else 0
        i_ease  = int(spec[1]) if len(spec) >= 2 else 0
        i_dir   = int(spec[2]) if len(spec) >= 3 else 0
    except ValueError:
        i_delay = i_ease = i_dir = 0

    # map delay/ease
    if "delay" in meta and isinstance(meta["delay"], list) and i_delay < len(meta["delay"]):
        delay = meta["delay"][i_delay]
    if "ease" in meta and isinstance(meta["ease"], list) and i_ease < len(meta["ease"]):
        ease = meta["ease"][i_ease]

    # map direction (B uses dict, A/C use list)
    if fam == "B" and isinstance(meta.get("direction"), dict):
        direction = meta["direction"].get(i_dir, np.nan)
    elif isinstance(meta.get("direction"), list) and i_dir < len(meta["direction"]):
        direction = meta["direction"][i_dir]

    return {
        "anim_family": fam,
        "anim_code": code,
        "anim_type": meta.get("type", np.nan),
        "delay": delay,
        "ease": ease,
        "direction": direction,
        "anim": True,
    }

def add_animation_columns(windows_df: pd.DataFrame) -> pd.DataFrame:
    """
    Make a copy of windows_df and add:
      anim_family, anim_code, anim_type, anim_delay, anim_ease, anim_direction, block_anim
    based on Task Id -> code mapping.
    """
    out = windows_df.copy()

    # ensure string task ids for mapping
    out["Task Id"] = out["Task Id"].astype(str).str.strip()

    # look up the code for each Task Id
    out["anim_code"] = out["Task Id"].map(task_mapping).fillna("Unknown")

    # decode each row
    decoded = out["anim_code"].apply(decode_from_code).apply(pd.Series)

    # attach
    for col in ["delay","ease","direction"]:
        out[col] = decoded[col]
    return out

windows_with_anim2 = add_animation_columns(features_task2)


windows_with_anim2.to_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/improving models/outputs2/windows_with_animation_details2.csv", index=False)
